# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [3]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [4]:
print("Spark version:", spark.version)
print("Spark application:", spark.sparkContext.appName)

Spark version: 4.2.0
Spark application: KafkaStreamingPrototype


In [5]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [6]:
print("Streaming:", raw_df.isStreaming)
raw_df.printSchema()

Streaming: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [8]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [13]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [14]:
print("Spark:", spark.version)
print("Aggregation exists:", "windowed_counts_df" in globals())
print("Streaming query exists:", "streaming_query" in globals())

Spark: 4.2.0
Aggregation exists: True
Streaming query exists: False


In [15]:
queries = spark.streams.active

print("Number of active queries:", len(queries))

for q in queries:
    print("Query ID:", q.id)
    print("Active:", q.isActive)
    print("Status:", q.status)

Number of active queries: 3
Query ID: f4380d1d-02c9-4911-83ef-6f7eeb6b2423
Active: True
Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}
Query ID: 9175e2a5-bf96-4007-9845-05681b034811
Active: True
Status: {'message': 'Getting offsets from KafkaV2[Subscribe[wikistreams]]', 'isDataAvailable': False, 'isTriggerActive': True}
Query ID: ba5c7d53-76cb-42ff-b95e-d291b0184033
Active: True
Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [17]:
queries = spark.streams.active

for i, q in enumerate(queries):
    print(f"\nQuery {i+1}")
    print("Active:", q.isActive)
    print("Last progress:", q.lastProgress)
    print("Exception:", q.exception())


Query 1
Active: True
Last progress: {
    "id": "f4380d1d-02c9-4911-83ef-6f7eeb6b2423",
    "runId": "6d49116f-14ef-40da-8e8c-6e5a6bcd5206",
    "name": null,
    "timestamp": "2026-10-17T21:30:36.693Z",
    "batchId": 1,
    "batchDuration": 20,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 20,
        "triggerExecution": 20
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 24,
            "numRowsUpdated": 0,
            "allUpdatesTimeMs": 2303,
            "numRowsRemoved": 0,
            "allRemovalsTimeMs": 0,
            "commitTimeMs": 49438,
            "memoryUsedBytes": 60016,
            "numRowsDroppedByWatermark": 0,
            "numShufflePartitions": 200,
            "numStateStoreInstances": 200,
            "customMetrics": {
                "SnapshotLastUploaded.partition_139_default": -1,
                "SnapshotLa

In [18]:
for q in spark.streams.active:
    q.stop()

print("Active queries:", len(spark.streams.active))

Active queries: 0


In [19]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

streaming_query = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

print("Streaming started:", streaming_query.isActive)

Streaming started: True


In [20]:
import time

time.sleep(15)

print("Active:", streaming_query.isActive)

progress = streaming_query.lastProgress

if progress:
    print("Batch:", progress["batchId"])
    print("Input rows:", progress["numInputRows"])
    print("Output rows:", progress["sink"]["numOutputRows"])
else:
    print("Waiting for first batch...")

Active: True
Batch: 1
Input rows: 0
Output rows: 0


In [21]:
import time

time.sleep(10)

print("Streaming active:", streaming_query.isActive)

if streaming_query.exception():
    print("Streaming error:", streaming_query.exception())

Streaming active: True


In [22]:
streaming_query.stop()

print("Streaming stopped:", not streaming_query.isActive)

Streaming stopped: True


In [23]:
# Overlapping Windows: 10-minute window every 5 minutes

overlapping_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "10 minutes", "5 minutes"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

print("Overlapping windows created!")

Overlapping windows created!


In [24]:
overlapping_query = overlapping_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

print("Overlapping Streaming:", overlapping_query.isActive)

Overlapping Streaming: True


In [25]:
import time

time.sleep(15)

progress = overlapping_query.lastProgress

print("Active:", overlapping_query.isActive)

if progress:
    print("Batch:", progress["batchId"])
    print("Input rows:", progress["numInputRows"])
    print("State rows:", progress["stateOperators"][0]["numRowsTotal"])
else:
    print("Waiting for first batch...")

Active: True
Batch: 1
Input rows: 0
State rows: 266


In [26]:
overlapping_query.stop()

print("Overlapping stopped:", not overlapping_query.isActive)

Overlapping stopped: True


In [29]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, LongType

# Schema for the edit size information
length_schema = StructType([
    StructField("old", LongType(), True),
    StructField("new", LongType(), True)
])

# Create a new schema without duplicate columns
schema_fields = [
    field for field in simple_wikimedia_schema.fields
    if field.name != "length"
]

edit_schema = StructType(
    schema_fields + [
        StructField("length", length_schema, True)
    ]
)

# Parse the JSON messages from Kafka
edit_size_df = raw_df.select(
    from_json(
        col("value").cast("string"),
        edit_schema
    ).alias("data")
).select("data.*")

# Calculate edit size difference
edit_size_df = edit_size_df.withColumn(
    "size_change",
    col("length.new") - col("length.old")
)

print("Edit size DataFrame created successfully!")

Edit size DataFrame created successfully!


In [30]:
from pyspark.sql.functions import avg, count, abs as spark_abs

edit_size_stats_df = edit_size_df \
    .withColumn("absolute_change", spark_abs(col("size_change"))) \
    .groupBy("bot") \
    .agg(
        count("*").alias("edit_count"),
        avg("size_change").alias("avg_size_change"),
        avg("absolute_change").alias("avg_absolute_change")
    )

print("Edit size aggregation created!")

Edit size aggregation created!


In [31]:
edit_size_query = edit_size_stats_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

print("Edit Size Streaming:", edit_size_query.isActive)

Edit Size Streaming: True


In [32]:
import time

time.sleep(15)

print("Active:", edit_size_query.isActive)

progress = edit_size_query.lastProgress

if progress:
    print("Batch:", progress["batchId"])
    print("Input rows:", progress["numInputRows"])
    print("Output rows:", progress["sink"]["numOutputRows"])
else:
    print("Waiting for first batch...")

Active: True
Batch: 1
Input rows: 0
Output rows: 0


In [33]:
for p in edit_size_query.recentProgress:
    print(
        "Batch:", p["batchId"],
        "| Input rows:", p["numInputRows"],
        "| Output rows:", p["sink"]["numOutputRows"]
    )

Batch: 0 | Input rows: 14117 | Output rows: 2
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0
Batch: 1 | Input rows: 0 | Output rows: 0


In [34]:
edit_size_query.stop()
print("Edit Size stopped:", not edit_size_query.isActive)

Edit Size stopped: True


In [35]:
# Filter: Human edits only
human_edits_df = parsed_df.filter(
    col("bot") == False
)

print("Human edits filter created!")
print("Streaming:", human_edits_df.isStreaming)

Human edits filter created!
Streaming: True


In [36]:
from pyspark.sql.functions import count

human_counts_df = human_edits_df \
    .groupBy("bot") \
    .agg(count("*").alias("total_edits"))

human_query = human_counts_df \
    .writeStream \
    .format("memory") \
    .queryName("human_edits_results") \
    .outputMode("complete") \
    .start()

human_query.processAllAvailable()

spark.sql(
    "SELECT * FROM human_edits_results"
).show(truncate=False)

+-----+-----------+
|bot  |total_edits|
+-----+-----------+
|false|10503      |
+-----+-----------+



In [37]:
human_query.stop()
print("Human filter stopped:", not human_query.isActive)

Human filter stopped: True
